In [ ]:
# Basic data manipulation libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# XGBoost model
from xgboost import XGBClassifier

# For randomized hyperparameter search
from scipy.stats import randint, uniform

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

In [ ]:
# File paths
train_transaction_path = "../data/train_transaction.csv"
train_identity_path = "../data/train_identity.csv"
test_transaction_path = "../data/test_transaction.csv"
test_identity_path = "../data/test_identity.csv"
sample_submission_path = "../data/sample_submission.csv"

# Load the dataset files
train_transaction = pd.read_csv(train_transaction_path)
train_identity = pd.read_csv(train_identity_path)
test_transaction = pd.read_csv(test_transaction_path)
test_identity = pd.read_csv(test_identity_path)
sample_submission = pd.read_csv(sample_submission_path)

print("Dataset files loaded successfully.")

In [ ]:
# Check the shape of each raw file
print("Train transaction shape:", train_transaction.shape)
print("Train identity shape:", train_identity.shape)
print("Test transaction shape:", test_transaction.shape)
print("Test identity shape:", test_identity.shape)
print("Sample submission shape:", sample_submission.shape)

In [ ]:
# Preview the main training transaction data
train_transaction.head()

In [ ]:
# Preview the training identity data
train_identity.head()

In [ ]:
# Preview the sample submission format
sample_submission.head()

In [ ]:
# Merge transaction and identity data using TransactionID
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test = test_transaction.merge(test_identity, on="TransactionID", how="left")

print("Merged train shape:", train.shape)
print("Merged test shape:", test.shape)

# Check whether the target column exists in the training data
print("\nTarget column exists in train:", "isFraud" in train.columns)

# Check whether TransactionID exists in both train and test
print("TransactionID exists in train:", "TransactionID" in train.columns)
print("TransactionID exists in test:", "TransactionID" in test.columns)

# Display first few rows of the merged training data
train.head()

In [ ]:
# Inspect the general structure of the merged datasets
print("Merged training data shape:", train.shape)
print("Merged test data shape:", test.shape)

print("\nFirst 10 columns:")
print(train.columns[:10].tolist())

print("\nLast 10 columns:")
print(train.columns[-10:].tolist())

print("\nData type counts in training data:")
print(train.dtypes.value_counts())

print("\nData type counts in test data:")
print(test.dtypes.value_counts())

In [ ]:
# Inspect the target variable and compare train/test columns

print("Target column name: isFraud")
print("\nTarget value counts:")
print(train["isFraud"].value_counts())

print("\nTarget value percentages:")
print(train["isFraud"].value_counts(normalize=True) * 100)

# Check train/test column alignment
train_feature_columns = set(train.drop(columns=["isFraud"]).columns)
test_columns = set(test.columns)

print("\nNumber of feature columns in train:", len(train_feature_columns))
print("Number of columns in test:", len(test_columns))

print("\nColumns in train features but not in test:")
print(train_feature_columns - test_columns)

print("\nColumns in test but not in train features:")
print(test_columns - train_feature_columns)

In [ ]:
# Standardize test identity column names
# Kaggle test identity columns may use '-' instead of '_' in column names such as id-01.
# We replace '-' with '_' to match the training column format.

test.columns = test.columns.str.replace("-", "_", regex=False)

# Re-check train/test column alignment after renaming
train_feature_columns = set(train.drop(columns=["isFraud"]).columns)
test_columns = set(test.columns)

print("Number of feature columns in train:", len(train_feature_columns))
print("Number of columns in test:", len(test_columns))

print("\nColumns in train features but not in test:")
print(train_feature_columns - test_columns)

print("\nColumns in test but not in train features:")
print(test_columns - train_feature_columns)

In [ ]:
# Visualize the target distribution

target_counts = train["isFraud"].value_counts().sort_index()
target_percentages = train["isFraud"].value_counts(normalize=True).sort_index() * 100

display(pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
}))

plt.figure(figsize=(6, 4))
target_counts.plot(kind="bar")
plt.title("Target Distribution: Fraud vs Non-Fraud Transactions")
plt.xlabel("isFraud")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Calculate missing value counts and percentages for the training data

train_missing_count = train.isnull().sum()
train_missing_percent = train.isnull().mean() * 100

train_missing_summary = pd.DataFrame({
    "missing_count": train_missing_count,
    "missing_percent": train_missing_percent
}).sort_values(by="missing_percent", ascending=False)

# Display columns with the highest missing value percentages
train_missing_summary.head(20)

In [ ]:
# Calculate missing value counts and percentages for the test data

test_missing_count = test.isnull().sum()
test_missing_percent = test.isnull().mean() * 100

test_missing_summary = pd.DataFrame({
    "missing_count": test_missing_count,
    "missing_percent": test_missing_percent
}).sort_values(by="missing_percent", ascending=False)

# Display columns with the highest missing value percentages
test_missing_summary.head(20)

In [ ]:
# Summarize missing value levels in train and test datasets

missing_summary_overview = pd.DataFrame({
    "dataset": ["train", "test"],
    "total_columns": [train.shape[1], test.shape[1]],
    "columns_with_missing": [
        (train.isnull().sum() > 0).sum(),
        (test.isnull().sum() > 0).sum()
    ],
    "columns_without_missing": [
        (train.isnull().sum() == 0).sum(),
        (test.isnull().sum() == 0).sum()
    ],
    "columns_above_50_percent_missing": [
        (train.isnull().mean() > 0.50).sum(),
        (test.isnull().mean() > 0.50).sum()
    ],
    "columns_above_90_percent_missing": [
        (train.isnull().mean() > 0.90).sum(),
        (test.isnull().mean() > 0.90).sum()
    ]
})

missing_summary_overview

In [ ]:
# Identify columns with more than 90% missing values in train and test

high_missing_threshold = 0.90

high_missing_train_cols = train.columns[train.isnull().mean() > high_missing_threshold].tolist()
high_missing_test_cols = test.columns[test.isnull().mean() > high_missing_threshold].tolist()

print("Number of train columns with more than 90% missing values:", len(high_missing_train_cols))
print("Train high-missing columns:")
print(high_missing_train_cols)

print("\nNumber of test columns with more than 90% missing values:", len(high_missing_test_cols))
print("Test high-missing columns:")
print(high_missing_test_cols)

# Columns that are high-missing in either train or test
high_missing_cols = sorted(list(set(high_missing_train_cols).union(set(high_missing_test_cols))))

print("\nTotal unique high-missing columns to consider:")
print(len(high_missing_cols))
print(high_missing_cols)

In [ ]:
# Drop columns with more than 90% missing values from both train and test

print("Shape before dropping high-missing columns:")
print("Train:", train.shape)
print("Test:", test.shape)

train_reduced = train.drop(columns=high_missing_cols)
test_reduced = test.drop(columns=high_missing_cols)

print("\nShape after dropping high-missing columns:")
print("Train:", train_reduced.shape)
print("Test:", test_reduced.shape)

# Re-check train/test feature alignment
train_reduced_features = set(train_reduced.drop(columns=["isFraud"]).columns)
test_reduced_features = set(test_reduced.columns)

print("\nColumns in train features but not in test:")
print(train_reduced_features - test_reduced_features)

print("\nColumns in test but not in train features:")
print(test_reduced_features - train_reduced_features)

In [ ]:
# Separate target, ID column, and feature columns

target_col = "isFraud"
id_col = "TransactionID"

X = train_reduced.drop(columns=[target_col])
y = train_reduced[target_col]

X_test = test_reduced.copy()

# Save TransactionID separately for final Kaggle submission
test_ids = X_test[id_col].copy()

# Remove TransactionID from modeling features
X = X.drop(columns=[id_col])
X_test = X_test.drop(columns=[id_col])

# Identify numerical and categorical columns
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Number of modeling features:", X.shape[1])
print("Number of numerical features:", len(numeric_cols))
print("Number of categorical features:", len(categorical_cols))

print("\nFirst 10 numerical columns:")
print(numeric_cols[:10])

print("\nCategorical columns:")
print(categorical_cols)

In [ ]:
# Check the number of unique values in categorical columns

categorical_unique_summary = pd.DataFrame({
    "column": categorical_cols,
    "unique_count": [X[col].nunique(dropna=True) for col in categorical_cols],
    "missing_percent": [X[col].isnull().mean() * 100 for col in categorical_cols]
}).sort_values(by="unique_count", ascending=False)

categorical_unique_summary

In [ ]:
# Inspect basic statistics of important numerical columns

important_numeric_cols = ["TransactionDT", "TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1"]

existing_important_numeric_cols = [col for col in important_numeric_cols if col in X.columns]

X[existing_important_numeric_cols].describe().T

In [ ]:
# Visualize the distribution of TransactionAmt

plt.figure(figsize=(8, 4))
train_reduced["TransactionAmt"].plot(kind="hist", bins=50)
plt.title("Distribution of Transaction Amount")
plt.xlabel("TransactionAmt")
plt.ylabel("Frequency")
plt.show()

# Also visualize the log-transformed transaction amount
plt.figure(figsize=(8, 4))
np.log1p(train_reduced["TransactionAmt"]).plot(kind="hist", bins=50)
plt.title("Distribution of Log-Transformed Transaction Amount")
plt.xlabel("log1p(TransactionAmt)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Inspect the TransactionDT variable

print("TransactionDT minimum:", train_reduced["TransactionDT"].min())
print("TransactionDT maximum:", train_reduced["TransactionDT"].max())

# Convert TransactionDT into approximate day and hour features for exploration
transaction_day = train_reduced["TransactionDT"] // (24 * 60 * 60)
transaction_hour = (train_reduced["TransactionDT"] // (60 * 60)) % 24

print("\nApproximate transaction day range:")
print("Minimum day:", transaction_day.min())
print("Maximum day:", transaction_day.max())

print("\nTransaction hour value counts:")
print(transaction_hour.value_counts().sort_index())

In [ ]:
# Create engineered features and check train/test alignment

def add_engineered_features(df):
    df = df.copy()
    
    # Log-transform transaction amount to reduce skewness
    df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])
    
    # Extract simple time-based features from TransactionDT
    df["TransactionDay"] = df["TransactionDT"] // (24 * 60 * 60)
    df["TransactionHour"] = (df["TransactionDT"] // (60 * 60)) % 24
    
    return df

# Apply the same feature engineering steps to train and test data
train_fe = add_engineered_features(train_reduced)
test_fe = add_engineered_features(test_reduced)

# Check shapes
print("Train shape before feature engineering:", train_reduced.shape)
print("Train shape after feature engineering:", train_fe.shape)

print("\nTest shape before feature engineering:", test_reduced.shape)
print("Test shape after feature engineering:", test_fe.shape)

# Check new features
new_features = ["TransactionAmt_log", "TransactionDay", "TransactionHour"]

print("\nNew features in train:")
print([col for col in new_features if col in train_fe.columns])

print("\nNew features in test:")
print([col for col in new_features if col in test_fe.columns])

# Check train/test feature alignment after feature engineering
train_fe_features = set(train_fe.drop(columns=["isFraud"]).columns)
test_fe_features = set(test_fe.columns)

print("\nNumber of train feature columns:", len(train_fe_features))
print("Number of test feature columns:", len(test_fe_features))

print("\nColumns in train features but not in test:")
print(train_fe_features - test_fe_features)

print("\nColumns in test but not in train features:")
print(test_fe_features - train_fe_features)

In [ ]:
# Define target and features

target_col = "isFraud"
id_col = "TransactionID"

y = train_fe[target_col]
X = train_fe.drop(columns=[target_col, id_col])

# Prepare final test features and save TransactionID for submission
test_ids = test_fe[id_col].copy()
X_test_final = test_fe.drop(columns=[id_col])

# Train-validation split with stratification
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test_final shape:", X_test_final.shape)

print("\nTraining target distribution (%):")
print(y_train.value_counts(normalize=True).sort_index() * 100)

print("\nValidation target distribution (%):")
print(y_val.value_counts(normalize=True).sort_index() * 100)

In [ ]:
# Identify numerical and categorical columns using only the training split

numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Number of numerical features:", len(numeric_cols))
print("Number of categorical features:", len(categorical_cols))

print("\nFirst 10 numerical features:")
print(numeric_cols[:10])

print("\nCategorical features:")
print(categorical_cols)

In [ ]:
# Build preprocessing pipelines for numerical and categorical features

# Preprocessing for tree-based models
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Preprocessing for Logistic Regression
numeric_transformer_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Common categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Preprocessor for tree-based models
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# Preprocessor for Logistic Regression
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_scaled, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

print("Preprocessing pipeline for tree-based models created successfully.")
print("Scaled preprocessing pipeline for Logistic Regression created successfully.")

In [ ]:
# Helper function to evaluate classification models

model_results = []

def evaluate_model(model_name, y_true, y_pred, y_proba):
    results = {
        "Model": model_name,
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }
    
    model_results.append(results)
    return pd.DataFrame([results])

# Baseline model: Logistic Regression

logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("model", LogisticRegression(
        solver="saga",
        penalty="l2",
        C=0.5,
        max_iter=80,
        tol=1e-2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=0
    ))
])

# Train the model
logreg_pipeline.fit(X_train, y_train)

# Validation predictions
logreg_val_pred = logreg_pipeline.predict(X_val)
logreg_val_proba = logreg_pipeline.predict_proba(X_val)[:, 1]

# Evaluate the model
evaluate_model(
    model_name="Logistic Regression",
    y_true=y_val,
    y_pred=logreg_val_pred,
    y_proba=logreg_val_proba
)

In [ ]:
# Baseline model: Decision Tree Classifier

dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ))
])

# Train the model
dt_pipeline.fit(X_train, y_train)

# Validation predictions
dt_val_pred = dt_pipeline.predict(X_val)
dt_val_proba = dt_pipeline.predict_proba(X_val)[:, 1]

# Evaluate the model
evaluate_model(
    model_name="Decision Tree",
    y_true=y_val,
    y_pred=dt_val_pred,
    y_proba=dt_val_proba
)

In [ ]:
# Ensemble model: Random Forest Classifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
rf_pipeline.fit(X_train, y_train)

# Validation predictions
rf_val_pred = rf_pipeline.predict(X_val)
rf_val_proba = rf_pipeline.predict_proba(X_val)[:, 1]

# Evaluate the model
evaluate_model(
    model_name="Random Forest",
    y_true=y_val,
    y_pred=rf_val_pred,
    y_proba=rf_val_proba
)

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

# Preprocessing for Gradient Boosting
# Rare categorical values are grouped to reduce one-hot encoding size.
categorical_transformer_gb = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ))
])

preprocessor_gb = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer_gb, categorical_cols)
    ]
)

# Boosting model: Gradient Boosting Classifier
gb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_gb),
    ("model", GradientBoostingClassifier(
        n_estimators=60,
        learning_rate=0.08,
        max_depth=2,
        min_samples_leaf=50,
        subsample=0.75,
        max_features="sqrt",
        n_iter_no_change=5,
        validation_fraction=0.1,
        tol=1e-3,
        random_state=42
    ))
])

# Balanced sample weights because GradientBoostingClassifier does not have class_weight
gb_sample_weight = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# Train the model
gb_pipeline.fit(X_train, y_train, model__sample_weight=gb_sample_weight)

# Validation predictions
gb_val_pred = gb_pipeline.predict(X_val)
gb_val_proba = gb_pipeline.predict_proba(X_val)[:, 1]

# Evaluate the model
evaluate_model(
    model_name="Gradient Boosting",
    y_true=y_val,
    y_pred=gb_val_pred,
    y_proba=gb_val_proba
)

In [ ]:
# Boosting model: XGBoost Classifier

import time

# Calculate class imbalance ratio for XGBoost
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / positive_count

print("scale_pos_weight:", scale_pos_weight)

# Preprocessing for XGBoost
# Rare categorical values are grouped to reduce one-hot encoding size.
categorical_transformer_xgb = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ))
])

preprocessor_xgb = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer_xgb, categorical_cols)
    ]
)

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_xgb),
    ("model", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

start_time = time.time()

# Train the model
xgb_pipeline.fit(X_train, y_train)

end_time = time.time()
print("Training time:", round((end_time - start_time) / 60, 2), "minutes")

# Validation predictions
xgb_val_pred = xgb_pipeline.predict(X_val)
xgb_val_proba = xgb_pipeline.predict_proba(X_val)[:, 1]

# Evaluate the model
evaluate_model(
    model_name="XGBoost",
    y_true=y_val,
    y_pred=xgb_val_pred,
    y_proba=xgb_val_proba
)

In [ ]:
pd.DataFrame(model_results).sort_values(by="ROC-AUC", ascending=False)

In [ ]:
# Hyperparameter tuning attempt for XGBoost

param_distributions_xgb = {
    "model__n_estimators": randint(100, 500),
    "model__learning_rate": uniform(0.01, 0.1),
    "model__max_depth": randint(3, 8),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.75, 0.20),
    "model__gamma": uniform(0, 0.5),
    "model__reg_alpha": uniform(0, 1),
    "model__reg_lambda": uniform(1, 4)
}

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions_xgb,
    n_iter=3,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=1,    
    verbose=2
)

start_time = time.time()

xgb_random_search.fit(X_train, y_train)

end_time = time.time()
print("Second tuning time:", round((end_time - start_time) / 60, 2), "minutes")

print("Best parameters from second tuning:")
print(xgb_random_search.best_params_)

print("\nBest cross-validation ROC-AUC from second tuning:")
print(xgb_random_search.best_score_)

In [ ]:
# Evaluate the tuned XGBoost model on the validation set

best_tuned_xgb_pipeline = xgb_random_search.best_estimator_

tuned_xgb_val_pred = best_tuned_xgb_pipeline.predict(X_val)
tuned_xgb_val_proba = best_tuned_xgb_pipeline.predict_proba(X_val)[:, 1]

evaluate_model(
    model_name="Tuned XGBoost",
    y_true=y_val,
    y_pred=tuned_xgb_val_pred,
    y_proba=tuned_xgb_val_proba
)

In [ ]:
# Final model comparison table

final_results_df = pd.DataFrame(model_results).sort_values(by="ROC-AUC", ascending=False)
final_results_df

In [ ]:
# Confusion matrix for the final tuned XGBoost model

cm = confusion_matrix(y_val, tuned_xgb_val_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Non-Fraud", "Fraud"],
    yticklabels=["Non-Fraud", "Fraud"]
)

plt.title("Confusion Matrix - Tuned XGBoost")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("Classification Report:")
print(classification_report(y_val, tuned_xgb_val_pred, zero_division=0))

In [ ]:
# ROC curve for the final tuned XGBoost model

fpr, tpr, thresholds = roc_curve(y_val, tuned_xgb_val_proba)
auc_score = roc_auc_score(y_val, tuned_xgb_val_proba)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"Tuned XGBoost AUC = {auc_score:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.title("ROC Curve - Tuned XGBoost")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

print("Validation ROC-AUC:", auc_score)

In [ ]:
# Generate fraud probability predictions for the Kaggle test set

test_fraud_proba = best_tuned_xgb_pipeline.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": test_fraud_proba
})

submission.head()

In [ ]:
# Save the submission file

submission_file_name = "submission_tuned_xgboost.csv"

submission.to_csv(submission_file_name, index=False)

print("Submission file saved as:", submission_file_name)
print("Submission shape:", submission.shape)

submission.head()

In [ ]:
# Check submission file format

saved_submission = pd.read_csv("submission_tuned_xgboost.csv")

print(saved_submission.shape)
print(saved_submission.columns.tolist())
print(saved_submission["isFraud"].min(), saved_submission["isFraud"].max())

saved_submission.head()